#### **Integrantes del grupo:**
Aleicer Vesga Rueda <br>
Jaider Morales Bautista


### 1. Esquema

![Diagrama de Entidad](images/diagrama2.jpg)

####Configura y evidencia la infraestructura en Databricks CE

#####1. Abrir la sección de cómputo

Desde la barra lateral izquierda del workspace, da clic en:

**Compute → Create compute**

Como se muestra en la imagen de la página 1 del documento, este es el punto de entrada para crear una nueva capacidad de cómputo (un servidor o nodo Spark en la nube). 

<br>

![](images/paso_uno.png)

#####**2. Asignar un nombre al cluster**

En el formulario de creación, introduce un nombre descriptivo para tu cluster.

Ejemplo: “Jaider Morales’s Cluster”

La siguiente imagen muestra cómo se ve esta sección.

![](images/paso_dos.png)



#####**3. Seleccionar el tipo de cluster y su entorno**

Ahora debes elegir qué tipo de cluster quieres levantar:

Cluster estándar (Standard)

Incluye:

	•	Scala
	•	Java
	•	Spark
	•	Configuración general para análisis, ETL y desarrollo

Este es el entorno más común cuando solo necesitas procesamiento distribuido y notebooks interactivos.

Toda esta parte se menciona explícitamente en la siguiente imagen.

![](images/paso_tres)

#####**Cluster para Machine Learning (ML)**

Incluye:

	•	Librerías preinstaladas para ML distribuido
	•	Paquetes como TensorFlow, XGBoost, MLflow, etc.
	•	Configuración optimizada para entrenamiento de modelos


![](images/paso_quinto)



#####4. Elegir la versión de Databricks Runtime

En “Databricks Runtime Version”, selecciona la versión que deseas usar.

Recomendaciones:

⭐ Versiones LTS (Long-Term Support)
	•	Mayor estabilidad
	•	Mejor recuperación ante fallos
	•	Soporte extendido
	•	Ideal para producción



⭐ Versiones recientes o no-LTS
	•	Nuevas funcionalidades
	•	Últimas mejoras de rendimiento

En tu ejemplo seleccionas una versión Scala 15.4 como se ve en la página 2 del archivo.

![](images/paso_sexto)

####**Elegir el tamaño del cluster**

Cada versión de Runtime muestra el tipo de máquina y su capacidad.
En tu ejemplo, seleccionas una configuración con 15GB de memoria, como aparece en la página 2.

Aquí también podrías definir:

	•	Número de workers
	•	Tipo de máquinas
	•	Autoescalado (opcional)



####**6. Crear el cluster**

Cuando todo esté configurado, simplemente haz clic en:

Create compute

Este es el paso final para que Databricks aprovisione tu servidor.
Tal como aparece en la última página del documento. <br>

![](images/paso_septimo.png)



## Creamos el catálogo y esquema

In [0]:
%sql
--Creamos el catálogo y el esquema
CREATE CATALOG IF NOT EXISTS salud_digital;
CREATE SCHEMA IF NOT EXISTS salud_digital.vital_sign;

--Se usa el esquema
USE salud_digital.vital_sign;

####Eliminamos la tabla si se ha creado previamente. 

In [0]:
%sql
DROP TABLE salud_digital.vital_sign.unified_measurements_table

#### DDL - Creamos la tabla con el esquema que tendra.

In [0]:
%sql
--Creamos el DDL para la tabla 
CREATE TABLE IF NOT EXISTS salud_digital.vital_sign.unified_measurements_table (
  patient_id              INT,
  heart_rate              INT,
  respiratory_rate        INT,
  timestamp               TIMESTAMP,
  body_temperature        DOUBLE,
  oxygen_saturation       DOUBLE,
  systolic_blood_pressure INT,
  diastolic_blood_pressure INT,
  age                     INT,
  gender                  STRING,
  weight_kg               DOUBLE,
  height_m                DOUBLE,
  derived_hrv             DOUBLE,
  derived_pulse_pressure  DOUBLE,
  derived_bmi             DOUBLE,
  derived_map             DOUBLE,
  risk_category           STRING
)
USING DELTA;

#### Vemos si la tabla si se ha creado

In [0]:
%sql
SHOW TABLES IN salud_digital.vital_sign;

database,tableName,isTemporary
vital_sign,unified_measurements_table,false
,_sqldf,true


#### Tabla ya con su definicion.

In [0]:
%sql
DESCRIBE TABLE salud_digital.vital_sign.unified_measurements_table;

col_name,data_type,comment
patient_id,int,null
heart_rate,int,null
respiratory_rate,int,null
timestamp,timestamp,null
body_temperature,double,null
oxygen_saturation,double,null
systolic_blood_pressure,int,null
diastolic_blood_pressure,int,null
age,int,null
gender,string,null


In [0]:
%sql
SHOW CREATE TABLE salud_digital.vital_sign.unified_measurements_table

createtab_stmt
"CREATE TABLE salud_digital.vital_sign.unified_measurements_table ( patient_id INT, heart_rate INT, respiratory_rate INT, timestamp TIMESTAMP, body_temperature DOUBLE, oxygen_saturation DOUBLE, systolic_blood_pressure INT, diastolic_blood_pressure INT, age INT, gender STRING, weight_kg DOUBLE, height_m DOUBLE, derived_hrv DOUBLE, derived_pulse_pressure DOUBLE, derived_bmi DOUBLE, derived_map DOUBLE, risk_category STRING) USING delta COLLATION 'UTF8_BINARY' TBLPROPERTIES ( 'delta.enableDeletionVectors' = 'true', 'delta.enableRowTracking' = 'true', 'delta.feature.appendOnly' = 'supported', 'delta.feature.deletionVectors' = 'supported', 'delta.feature.domainMetadata' = 'supported', 'delta.feature.invariants' = 'supported', 'delta.feature.rowTracking' = 'supported', 'delta.minReaderVersion' = '3', 'delta.minWriterVersion' = '7', 'delta.parquet.compression.codec' = 'zstd')"


####Comentario sobre el proceso de ingesta manual del dataset

El dataset seleccionado contiene nombres de columnas con espacios, paréntesis y caracteres especiales, por ejemplo: <br>


	•	Patient ID
	•	Heart Rate
	•	Weight (kg)
	•	Risk Category

Estas convenciones no son compatibles con las reglas del motor Delta Lake, el cual no permite crear tablas con este tipo de nombres a menos que se habilite column mapping, una funcionalidad avanzada que no está configurada en el entorno del curso.

	

Por esta razón, no fue posible utilizar el mismo procedimiento usado en clases, que consistía en:
<br>

	1.	Crear una tabla Delta con los mismos nombres del CSV.
	2.	Cargar los datos con COPY INTO ... BY POSITION.

En nuestro caso, este método produce errores como:

[DELTA_INVALID_CHARACTERS_IN_COLUMN_NAMES] Invalid column names

####Procedimiento utilizado

Para garantizar una ingesta correcta y mantener la integridad del modelo, se optó por un procedimiento alternativo dentro de las herramientas ofrecidas por Databricks: <br>

1. Se ingresó a Data Ingestion en Databricks.
2. Se utilizó la opción Create or Modify Table from File Upload.
3. Se cargó el archivo CSV directamente desde la interfaz gráfica.
4. Se seleccionó: <br>
  
-   El catálogo correspondiente, 
- luego el schema,
- y finalmente la tabla que debía ser creada o modificada.
<br>

5. Databricks realizó automáticamente el procesamiento de los nombres de columnas inválidos, permitiendo la creación de la tabla sin conflictos y generando el esquema compatible.

Este flujo es completamente válido dentro del entorno Databricks, y se convierte en la alternativa correcta cuando los nombres de columnas del archivo fuente no cumplen con las reglas del motor Delta.

![](images/creacion_modificacion_tabla)

In [0]:
%sql
DESCRIBE TABLE salud_digital.vital_sign.unified_measurements_table

col_name,data_type,comment
Patient ID,bigint,null
Heart Rate,bigint,null
Respiratory Rate,bigint,null
Timestamp,timestamp,null
Body Temperature,double,null
Oxygen Saturation,double,null
Systolic Blood Pressure,bigint,null
Diastolic Blood Pressure,bigint,null
Age,bigint,null
Gender,string,null


### Volumen
![](images/volumen.png)

####Renombrando las columnas con buenas practicas

In [0]:
%sql
ALTER TABLE salud_digital.vital_sign.unified_measurements_table
RENAME COLUMN `Patient ID` TO patient_id;

In [0]:
%sql
ALTER TABLE salud_digital.vital_sign.unified_measurements_table
RENAME COLUMN `Heart Rate` TO heart_rate;

ALTER TABLE salud_digital.vital_sign.unified_measurements_table
RENAME COLUMN `Respiratory Rate` TO respiratory_rate;

ALTER TABLE salud_digital.vital_sign.unified_measurements_table
RENAME COLUMN `Body Temperature` TO body_temperature;

ALTER TABLE salud_digital.vital_sign.unified_measurements_table
RENAME COLUMN `Oxygen Saturation` TO oxygen_saturation;

ALTER TABLE salud_digital.vital_sign.unified_measurements_table
RENAME COLUMN `Systolic Blood Pressure` TO systolic_blood_pressure;

ALTER TABLE salud_digital.vital_sign.unified_measurements_table
RENAME COLUMN `Diastolic Blood Pressure` TO diastolic_blood_pressure;

ALTER TABLE salud_digital.vital_sign.unified_measurements_table
RENAME COLUMN `Weight (kg)` TO weight_kg;

ALTER TABLE salud_digital.vital_sign.unified_measurements_table
RENAME COLUMN `Height (m)` TO height_m;

ALTER TABLE salud_digital.vital_sign.unified_measurements_table
RENAME COLUMN `Risk Category` TO risk_category;

#####Cambiamos los tipos de datos BIGINT por la version mas recomendada INT + DOUBLE, lo cual es mas estandar, mas limpia y eficiente.

In [0]:
%sql
CREATE OR REPLACE TABLE salud_digital.vital_sign.unified_measurements_table_clean
USING DELTA
AS
SELECT
  CAST(patient_id AS INT)                AS patient_id,
  CAST(heart_rate AS INT)                AS heart_rate,
  CAST(respiratory_rate AS INT)          AS respiratory_rate,
  CAST(Timestamp AS TIMESTAMP)           AS timestamp,
  CAST(body_temperature AS DOUBLE)       AS body_temperature,
  CAST(oxygen_saturation AS DOUBLE)      AS oxygen_saturation,
  CAST(systolic_blood_pressure AS INT)   AS systolic_blood_pressure,
  CAST(diastolic_blood_pressure AS INT)  AS diastolic_blood_pressure,
  CAST(Age AS INT)                       AS age,
  CAST(Gender AS STRING)                 AS gender,
  CAST(weight_kg AS DOUBLE)              AS weight_kg,
  CAST(height_m AS DOUBLE)               AS height_m,
  CAST(Derived_HRV AS DOUBLE)            AS derived_hrv,
  CAST(Derived_Pulse_Pressure AS DOUBLE) AS derived_pulse_pressure,
  CAST(Derived_BMI AS DOUBLE)            AS derived_bmi,
  CAST(Derived_MAP AS DOUBLE)            AS derived_map,
  CAST(risk_category AS STRING)          AS risk_category
FROM salud_digital.vital_sign.unified_measurements_table;

num_affected_rows,num_inserted_rows


In [0]:
%sql
DESCRIBE TABLE salud_digital.vital_sign.unified_measurements_table_clean;

col_name,data_type,comment
patient_id,int,null
heart_rate,int,null
respiratory_rate,int,null
timestamp,timestamp,null
body_temperature,double,null
oxygen_saturation,double,null
systolic_blood_pressure,int,null
diastolic_blood_pressure,int,null
age,int,null
gender,string,null


In [0]:
%sql
DROP TABLE salud_digital.vital_sign.unified_measurements_table;

In [0]:
%sql
ALTER TABLE salud_digital.vital_sign.unified_measurements_table_clean
RENAME TO salud_digital.vital_sign.unified_measurements_table;

####Validaciones

#####1) Metadatos de la tabla

In [0]:
%sql
DESCRIBE TABLE salud_digital.vital_sign.unified_measurements_table;

col_name,data_type,comment
patient_id,int,null
heart_rate,int,null
respiratory_rate,int,null
timestamp,timestamp,null
body_temperature,double,null
oxygen_saturation,double,null
systolic_blood_pressure,int,null
diastolic_blood_pressure,int,null
age,int,null
gender,string,null


Verificar estructura física: nombres reales de columnas, tipos de dato y orden. Sirve para confirmar que la ingesta creó el esquema esperado.

In [0]:
%sql
SHOW CREATE TABLE salud_digital.vital_sign.unified_measurements_table;

createtab_stmt
"CREATE TABLE salud_digital.vital_sign.unified_measurements_table ( patient_id INT, heart_rate INT, respiratory_rate INT, timestamp TIMESTAMP, body_temperature DOUBLE, oxygen_saturation DOUBLE, systolic_blood_pressure INT, diastolic_blood_pressure INT, age INT, gender STRING, weight_kg DOUBLE, height_m DOUBLE, derived_hrv DOUBLE, derived_pulse_pressure DOUBLE, derived_bmi DOUBLE, derived_map DOUBLE, risk_category STRING) USING delta COLLATION 'UTF8_BINARY' TBLPROPERTIES ( 'delta.enableDeletionVectors' = 'true', 'delta.feature.appendOnly' = 'supported', 'delta.feature.deletionVectors' = 'supported', 'delta.feature.invariants' = 'supported', 'delta.minReaderVersion' = '3', 'delta.minWriterVersion' = '7', 'delta.parquet.compression.codec' = 'zstd')"


Propósito: ver el DDL exacto con propiedades Delta/ubicación. Valida que la tabla es Delta y quedó en el catálogo/esquema correctos.

#####2) Conteo total + muestra rápida

In [0]:
%sql
SELECT COUNT(*) AS total_filas
FROM salud_digital.vital_sign.unified_measurements_table;

total_filas
200020


Propósito: validar número de filas cargadas (compararlo con filas del CSV). Si difiere mucho, algo falló.

In [0]:
%sql
SELECT *
FROM salud_digital.vital_sign.unified_measurements_table
LIMIT 10;

patient_id,heart_rate,respiratory_rate,timestamp,body_temperature,oxygen_saturation,systolic_blood_pressure,diastolic_blood_pressure,age,gender,weight_kg,height_m,derived_hrv,derived_pulse_pressure,derived_bmi,derived_map,risk_category
1,60,12,2024-07-19T21:53:45.729Z,36.861707108012936,95.70204560529264,124,86,37,Female,91.54161781042703,1.6793511495386064,0.12103285740470844,38.0,32.45903107193736,98.66666666666667,High Risk
2,63,18,2024-07-19T21:52:45.729Z,36.51163284237428,96.68941321884793,126,84,77,Male,50.704921363786234,1.992546278864286,0.11706154720647544,42.0,12.77124626305356,98.0,High Risk
3,63,15,2024-07-19T21:51:45.729Z,37.05204858016934,98.50826478317362,131,78,68,Female,90.3167596936639,1.770227684332464,0.0531999581506069,53.0,28.821069406784925,95.66666666666666,Low Risk
4,99,16,2024-07-19T21:50:45.729Z,36.654747504854214,95.01180149205474,118,72,41,Female,96.00618785155217,1.833629095455426,0.06447467683467452,46.0,28.554610608265143,87.33333333333333,High Risk
5,69,16,2024-07-19T21:49:45.729Z,36.97509754422735,98.6237917407539,138,76,25,Female,56.020005821418636,1.8664189298722969,0.11848422142020956,62.0,16.08143828760833,96.66666666666666,High Risk
6,79,12,2024-07-19T21:48:45.729Z,36.884979062580214,95.9871292033958,130,70,22,Male,79.86993283621617,1.9223336907147646,0.10396315743710202,60.0,21.61353304286171,90.0,Low Risk
7,81,17,2024-07-19T21:47:45.729Z,37.273639584377314,99.45671576538214,118,84,43,Male,57.84656504041972,1.831483806988997,0.05588465241439729,34.0,17.245326017670862,95.33333333333333,High Risk
8,96,15,2024-07-19T21:46:45.729Z,36.85263343367673,97.1241246758814,135,77,72,Female,71.758971665616,1.6038878729923471,0.07341337269827657,58.0,27.895117756083998,96.33333333333333,High Risk
9,83,12,2024-07-19T21:45:45.729Z,36.044191421051224,98.58449733479578,111,84,50,Male,79.29533165339934,1.6727351544521356,0.09851956280275856,27.0,28.339569682837293,93.0,Low Risk
10,66,15,2024-07-19T21:44:45.729Z,36.95717841280944,97.91626727887889,131,77,61,Male,53.923400264374514,1.8963808200748644,0.0813638585935069,54.0,14.994298811649214,95.0,High Risk


Propósito: inspección visual: confirmar que los datos “se ven bien”, que no hay filas corridas por mal orden, etc.

#####3) Descripción de datos (estadísticos básicos)

######3.1 Estadísticas para variables numéricas clave

In [0]:
%sql
SELECT
  COUNT(*) AS n,
  ROUND(AVG(heart_rate), 2) AS avg_heart_rate,
  MIN(heart_rate) AS min_heart_rate,
  MAX(heart_rate) AS max_heart_rate,
  ROUND(AVG(body_temperature), 2) AS avg_temp,
  MIN(body_temperature) AS min_temp,
  MAX(body_temperature) AS max_temp,
  ROUND(AVG(oxygen_saturation), 2) AS avg_ox_sat,
  MIN(oxygen_saturation) AS min_ox_sat,
  MAX(oxygen_saturation) AS max_ox_sat
FROM salud_digital.vital_sign.unified_measurements_table;

n,avg_heart_rate,min_heart_rate,max_heart_rate,avg_temp,min_temp,max_temp,avg_ox_sat,min_ox_sat,max_ox_sat
200020,79.53,60,99,36.75,36.00000442562442,37.49999177587785,97.5,95.0000066683982,99.99996250272243


Propósito: validar rangos plausibles (ej. temperatura ~35–40°C, oxigenación ~90–100). Detecta valores extremos o columnas mal tipeadas.

#####4) Validaciones con SELECT + GRUPO BY

In [0]:
%sql
SELECT
  risk_category,
  COUNT(*) AS total,
  ROUND(
    100 * COUNT(*) / SUM(COUNT(*)) OVER (),
    2
  ) AS porcentaje
FROM salud_digital.vital_sign.unified_measurements_table
GROUP BY risk_category
ORDER BY total DESC;

risk_category,total,porcentaje
High Risk,105115,52.55
Low Risk,94905,47.45


Propósito: validar que la variable categórica existe, tiene valores esperados y distribución lógica.

####Ventajas y desventajas: SQL vs Spark

| Herramienta  | Ventajas | Desventajas |
|--------------|----------|-------------|
| **SQL** | - SQL es muy fácil de aprender porque usa un lenguaje claro y estructurado para consultar datos. <br> - Funciona muy bien para bases de datos que no son demasiado grandes y donde se necesita precisión en las consultas. <br> - Es estable, confiable y está presente en casi todos los sistemas de gestión de datos. <br> - Permite realizar consultas rápidas sin necesidad de grandes configuraciones. <br> - Es ideal para trabajos donde se requiere integridad, orden y relaciones entre tablas. | - Cuando el volumen de datos crece mucho, SQL puede volverse lento y difícil de escalar. <br> - Depende de un solo servidor o máquina, lo que limita el rendimiento. <br> - No está pensado para procesar datos distribuidos o en tiempo real. <br> - Puede ser menos flexible cuando se trabajan datos no estructurados o de diferentes fuentes. |
| **Apache Spark** | - Está diseñado para manejar enormes cantidades de datos de manera rápida gracias al procesamiento distribuido. <br> - Puede trabajar con datos en diferentes formatos y ubicaciones, no solo en bases de datos tradicionales. <br> - Permite análisis complejos, aprendizaje automático y tareas avanzadas que SQL no puede hacer solo. <br> - Es ideal para proyectos de Big Data, análisis en tiempo real y ambientes empresariales. | - RRequiere más recursos de hardware y un entorno preparado para aprovecharlo bien. <br> - Su curva de aprendizaje es mayor, ya que involucra conceptos más avanzados y distintos a SQL. <br> - Para tareas pequeñas puede ser demasiado pesado y no justificar su uso. <br> - Su configuración inicial y mantenimiento pueden ser más complejos que trabajar con una base SQL tradicional. |